# Model Interpretation & Explainability

Welcome to the assignment for Model Interpretation & Explainability.

Building a model that generalises well is only half the job. The other half is being able to **explain** what the model has learned : which features drive its predictions, how much each feature contributes to a specific outcome, and whether the model's reasoning aligns with domain knowledge.

In this assignment you will apply a progression of interpretation techniques: from the simple (reading linear model coefficients) to the sophisticated (SHAP Shapley values and LIME local explanations). You will work with both global explanations (what the model learned overall) and local explanations (why was *this particular prediction* made?).

**What You Will Do in This Assignment**

* Interpret **linear model coefficients** and understand feature impact.
* Visualise and extract **decision rules from tree models**.
* Compare **permutation importance, SHAP, and LIME** across models.
* Create SHAP **summary, dependence, and force plots**.
* Implement **permutation importance from scratch**.
* Build a comprehensive **interpretation report**.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

*   All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

*   In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

*   You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

*   Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission.

## Table of Contents

1.  [Introduction to Model Interpretation](#1)
2.  [Linear Model Interpretation](#2)
    *   [Exercise 1: Coefficient Analysis](#ex-1)
3.  [Tree Model Interpretation](#3)
    *   [Exercise 2: Decision Tree Visualisation](#ex-2)
4.  [Feature Importance Methods](#4)
    *   [Exercise 3: Permutation Importance](#ex-3)
5.  [SHAP Deep Dive](#5)
    *   [Exercise 4: SHAP Analysis](#ex-4)
6.  [Local Explanations with LIME](#6)
    *   [Exercise 5: LIME Explanations](#ex-5)
7.  [From-Scratch Implementation](#7)
    *   [Exercise 6: Permutation Importance from Scratch](#ex-6)
8.  [Conclusion](#8)

<a name='1'></a>
## 1. Introduction to Model Interpretation

**Model interpretation** is the degree to which a human can understand the cause of a decision made by a machine learning model. It answers questions like:
- Which features matter most to the model?
- How does changing a feature value affect the prediction?
- Why did the model make *this specific prediction* for *this specific instance*?

### Interpretability vs Explainability

| Term | Scope | Example |
|---|---|---|
| **Interpretability** | Global : understanding how the model works overall | Feature importances, model coefficients |
| **Explainability** | Local : understanding a single prediction | SHAP force plot, LIME explanation |

### Key Techniques

| Method | Scope | Model-Agnostic? | Description |
|---|---|---|---|
| Coefficients | Global | No (linear only) | Direct weight interpretation |
| Tree rules | Global/Local | No (trees only) | Decision path visualisation |
| Permutation Importance | Global | Yes | Score drop when feature is shuffled |
| SHAP | Global + Local | Yes | Shapley value game-theoretic attribution |
| LIME | Local | Yes | Local linear approximation |

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import unittests
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, r2_score
from sklearn.base import clone

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Install with: pip install shap")

try:
    from lime import lime_tabular
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("LIME not installed. Install with: pip install lime")

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

### Load Datasets

We will use two well-known datasets:
- **Breast Cancer** (classification, 30 features) : predict malignant vs benign tumour.
- **Diabetes** (regression, 10 features) : predict disease progression one year after baseline.

In [ ]:
# ── Classification dataset: Breast Cancer ──────────────────────────────────
cancer = load_breast_cancer()
X_clf = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_clf = cancer.target   # 0 = malignant, 1 = benign

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# ── Regression dataset: Diabetes ────────────────────────────────────────────
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = diabetes.target   # disease progression

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Breast Cancer  : train: {X_clf_train.shape}, test: {X_clf_test.shape}")
print(f"Diabetes       : train: {X_reg_train.shape}, test: {X_reg_test.shape}")

<a name='2'></a>
## 2. Linear Model Interpretation

Linear models (logistic regression, ridge regression) are inherently interpretable through their **coefficients**. Each coefficient represents the change in the output per unit change in the corresponding feature, all else being equal.

For standardised features the magnitude of a coefficient directly reflects the feature's importance.

### Key rules
- **Positive coefficient** → higher feature value pushes prediction upward.
- **Negative coefficient** → higher feature value pushes prediction downward.
- **|coefficient| magnitude** → strength of the feature's influence.
- Must **standardise** features before comparing coefficient magnitudes.

<a name='ex-1'></a>
### Exercise 1: Linear Model Coefficient Analysis

**Your Task:**

Implement `interpret_linear_model` that:
1. Fits a standardised `Ridge` regression pipeline on `(X_train, y_train)`.
2. Extracts the coefficients from the fitted pipeline.
3. Returns a `pd.DataFrame` with columns `feature` and `coefficient`, **sorted by absolute coefficient value descending**.

**Arguments:**
- `X_train`, `y_train` : training data (DataFrame + array).
- `X_test`, `y_test` : test data for computing R² score.
- `alpha` : Ridge regularisation strength (default `1.0`).

**Returns:** `(coef_df, r2_score_value)` : DataFrame and float.

In [ ]:
# GRADED FUNCTION: interpret_linear_model
def interpret_linear_model(X_train, y_train, X_test, y_test, alpha=1.0):
    """
    Fit a Ridge regression pipeline (StandardScaler → Ridge) and extract
    sorted coefficient analysis.

    Parameters
    ----------
    X_train : pd.DataFrame
    y_train : array-like
    X_test  : pd.DataFrame
    y_test  : array-like
    alpha   : float, Ridge regularisation strength

    Returns
    -------
    coef_df : pd.DataFrame  columns=['feature', 'coefficient'], sorted by |coef|
    r2      : float         R² on test set
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###
    return coef_df, r2

In [ ]:
# Test interpret_linear_model on Diabetes dataset
coef_df, r2 = interpret_linear_model(
    X_reg_train, y_reg_train, X_reg_test, y_reg_test, alpha=1.0
)

print(f"Ridge R² on test set: {r2:.4f}")
print("\nTop 5 most influential features:")
print(coef_df.head())

# Visualise all coefficients
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0' if c >= 0 else '#C44E52' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Coefficient value (standardised features)')
ax.set_title('Ridge Regression Coefficients : Diabetes Dataset\n'
             '(Blue = positive influence, Red = negative influence)',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_1(interpret_linear_model)

<a name='3'></a>
## 3. Tree Model Interpretation

Decision trees are among the most interpretable models because every prediction can be traced as a path through a sequence of human-readable if/else rules.

Key concepts:
- **Feature importance (impurity-based):** mean decrease in impurity (Gini/entropy) weighted by the number of samples that pass through the node.
- **Decision path:** the sequence of nodes a specific instance traverses to reach its predicted class.
- **`export_text`:** converts a tree to a text-based rule set.

<a name='ex-2'></a>
### Exercise 2: Decision Tree Visualisation & Rule Extraction

**Your Task:**

Implement `interpret_tree_model` that:
1. Fits a `DecisionTreeClassifier(max_depth=4, random_state=42)` on `(X_train, y_train)`.
2. Extracts **feature importances** into a `pd.DataFrame` with columns `feature` and `importance`, sorted descending.
3. Extracts text rules using `export_text`.
4. Computes test accuracy.

**Returns:** `(importance_df, text_rules, accuracy)`

In [ ]:
# GRADED FUNCTION: interpret_tree_model
def interpret_tree_model(X_train, y_train, X_test, y_test, max_depth=4):
    """
    Fit a DecisionTreeClassifier and extract feature importances and rules.

    Returns
    -------
    importance_df : pd.DataFrame  columns=['feature', 'importance'], sorted desc
    text_rules    : str           human-readable decision rules
    accuracy      : float         test accuracy
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###
    return importance_df, text_rules, accuracy

In [ ]:
# Test interpret_tree_model on Breast Cancer
imp_df, rules, acc = interpret_tree_model(
    X_clf_train, y_clf_train, X_clf_test, y_clf_test, max_depth=4
)

print(f"Decision Tree accuracy: {acc:.4f}")
print("\nTop 5 most important features:")
print(imp_df.head())

print("\nDecision rules (first 600 chars):")
print(rules[:600])

# Plot tree
dt_vis = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_vis.fit(X_clf_train, y_clf_train)

fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt_vis, feature_names=list(X_clf_train.columns),
          class_names=cancer.target_names,
          filled=True, rounded=True, ax=ax, fontsize=8)
ax.set_title('Decision Tree (max_depth=3) : Breast Cancer', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_2(interpret_tree_model)

<a name='4'></a>
## 4. Feature Importance Methods

Different importance methods often tell different stories. Comparing them reveals which features are **robustly** important across interpretations.

| Method | How it works | Strengths | Weaknesses |
|---|---|---|---|
| **Impurity (MDI)** | Mean decrease in Gini/entropy | Fast, built-in | Biased toward high-cardinality features |
| **Permutation** | Score drop after shuffling feature | Model-agnostic, unbiased | Slow, correlated features split importance |
| **SHAP** | Shapley values from game theory | Consistent, additive | Computationally expensive |

<a name='ex-3'></a>
### Exercise 3: Permutation Importance Analysis

**Your Task:**

Implement `compute_permutation_importance` that:
1. Fits a `RandomForestClassifier(n_estimators=100, random_state=42)` on training data.
2. Computes **permutation importance** on the **test set** using `sklearn.inspection.permutation_importance` with `n_repeats=10, random_state=42`.
3. Returns a `pd.DataFrame` with columns `feature`, `importance_mean`, `importance_std`, sorted by `importance_mean` descending.
4. Also returns test accuracy.

**Returns:** `(perm_df, accuracy)`

In [ ]:
# GRADED FUNCTION: compute_permutation_importance
def compute_permutation_importance(X_train, y_train, X_test, y_test):
    """
    Fit RandomForest and compute permutation importance on the test set.

    Returns
    -------
    perm_df  : pd.DataFrame  columns=['feature', 'importance_mean', 'importance_std']
    accuracy : float         test accuracy
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###
    return perm_df, accuracy

In [ ]:
# Test compute_permutation_importance on Breast Cancer
perm_df, rf_acc = compute_permutation_importance(
    X_clf_train, y_clf_train, X_clf_test, y_clf_test
)

print(f"Random Forest accuracy: {rf_acc:.4f}")
print("\nTop 10 features by permutation importance:")
print(perm_df.head(10).to_string(index=False))

# Horizontal bar chart with error bars
fig, ax = plt.subplots(figsize=(10, 7))
top10 = perm_df.head(10)
ax.barh(top10['feature'][::-1], top10['importance_mean'][::-1],
        xerr=top10['importance_std'][::-1],
        color='#4C72B0', alpha=0.85, edgecolor='white', capsize=4)
ax.set_xlabel('Mean accuracy decrease after permutation')
ax.set_title('Permutation Importance (top 10) : Breast Cancer', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_3(compute_permutation_importance)

<a name='5'></a>
## 5. SHAP Deep Dive

**SHAP (SHapley Additive exPlanations)** assigns each feature a Shapley value : the average marginal contribution of that feature across all possible subsets of features.

### Key Properties
- **Additivity:** SHAP values for all features sum exactly to the prediction minus the baseline (`E[f(X)]`).
- **Consistency:** if a model changes such that a feature has more impact, its SHAP value increases.
- **Local accuracy:** `f(x) = base_value + sum(shap_values)`.

### SHAP Plot Types
| Plot | Purpose |
|---|---|
| **Summary plot** | Beeswarm showing feature impact distribution across all samples |
| **Bar plot** | Mean absolute SHAP value per feature (global importance) |
| **Dependence plot** | SHAP value vs feature value (shows non-linearities) |
| **Force plot** | Single prediction breakdown into feature contributions |

<a name='ex-4'></a>
### Exercise 4: SHAP Analysis

**Your Task:**

Implement `compute_shap_values` that:
1. Fits a `GradientBoostingClassifier(n_estimators=100, random_state=42)` on training data.
2. Creates a `shap.TreeExplainer` for the fitted model.
3. Computes SHAP values for `X_test`.
4. Returns `(explainer, shap_values, accuracy)`.

**Note:** if `shap` is not installed, return `(None, None, accuracy)` so the rest of the notebook still runs.

In [ ]:
# GRADED FUNCTION: compute_shap_values
def compute_shap_values(X_train, y_train, X_test, y_test):
    """
    Fit GradientBoostingClassifier and compute SHAP values using TreeExplainer.

    Returns
    -------
    explainer   : shap.TreeExplainer  (or None if shap not available)
    shap_values : np.ndarray or list  SHAP values for X_test
    accuracy    : float               test accuracy
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###
    return explainer, shap_values, accuracy

In [ ]:
# Test compute_shap_values
explainer, shap_vals, gb_acc = compute_shap_values(
    X_clf_train, y_clf_train, X_clf_test, y_clf_test
)
print(f"Gradient Boosting accuracy: {gb_acc:.4f}")

if SHAP_AVAILABLE and shap_vals is not None:
    print(f"SHAP values shape: {np.array(shap_vals).shape}")

    # Summary beeswarm plot
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_vals, X_clf_test, show=False)
    plt.title('SHAP Summary Plot : Breast Cancer (Gradient Boosting)', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Bar plot: mean |SHAP|
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_clf_test, plot_type='bar', show=False)
    plt.title('Mean |SHAP| Feature Importance', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Dependence plot for top feature
    top_feat = pd.DataFrame({
        'feature': list(X_clf_test.columns),
        'mean_shap': np.abs(shap_vals).mean(0)
    }).sort_values('mean_shap', ascending=False).iloc[0]['feature']
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(top_feat, shap_vals, X_clf_test, show=False)
    plt.title(f'SHAP Dependence Plot : {top_feat}', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("SHAP not available : skipping plots. Install with: pip install shap")

In [ ]:
unittests.exercise_4(compute_shap_values)

<a name='6'></a>
## 6. Local Explanations with LIME

**LIME (Local Interpretable Model-agnostic Explanations)** explains individual predictions by fitting a simpler interpretable model (linear) in the **local neighbourhood** of the instance being explained.

### How LIME Works
1. Select the instance to explain.
2. Generate **perturbed samples** around it.
3. Get model predictions for all perturbed samples.
4. Weight samples by their **proximity** to the original instance.
5. Fit a **weighted linear model** on the perturbed samples.
6. The linear model's coefficients are the explanation.

### Global vs Local
| SHAP | LIME |
|---|---|
| Exact game-theoretic attribution | Approximate local linear fit |
| Globally consistent | Only valid locally |
| Slower (tree: fast, neural: slow) | Fast for any model |
| Can capture feature interactions | Assumes local linearity |

<a name='ex-5'></a>
### Exercise 5: LIME Explanations

**Your Task:**

Implement `compute_lime_explanation` that:
1. Creates a `LimeTabularExplainer` using `X_train` (numpy array), `feature_names`, `class_names`, and `mode='classification'`.
2. Generates an explanation for instance `X_test.iloc[instance_idx]` using `num_features=10`.
3. Extracts the explanation as a list of `(feature_name, weight)` tuples sorted by **absolute weight descending**.
4. Returns `(explanation_list, lime_explainer)`.

**Returns:** `([(feature, weight), ...], explainer_object)`

In [ ]:
# GRADED FUNCTION: compute_lime_explanation
def compute_lime_explanation(model, X_train, X_test, feature_names,
                              class_names, instance_idx=0):
    """
    Generate a LIME explanation for a single instance.

    Parameters
    ----------
    model         : fitted sklearn classifier
    X_train       : pd.DataFrame  training data (used to build the explainer)
    X_test        : pd.DataFrame  test data
    feature_names : list of str
    class_names   : list of str
    instance_idx  : int           index of the test instance to explain

    Returns
    -------
    explanation_list : list of (feature_name, weight) sorted by |weight| desc
    lime_explainer   : LimeTabularExplainer object (or None if LIME unavailable)
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###
    return explanation_list, lime_explainer

In [ ]:
# Fit a GradientBoosting model for LIME
gb_lime = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_lime.fit(X_clf_train, y_clf_train)

instance_idx = 5   # explain the 6th test instance

lime_list, lime_exp_obj = compute_lime_explanation(
    model         = gb_lime,
    X_train       = X_clf_train,
    X_test        = X_clf_test,
    feature_names = list(X_clf_train.columns),
    class_names   = list(cancer.target_names),
    instance_idx  = instance_idx
)

true_label = y_clf_test.iloc[instance_idx] if hasattr(y_clf_test, 'iloc') else y_clf_test[instance_idx]
pred_label = gb_lime.predict(X_clf_test.iloc[[instance_idx]])[0]
print(f"Instance {instance_idx}: true={cancer.target_names[true_label]}, "
      f"predicted={cancer.target_names[pred_label]}")

if lime_list:
    print("\nTop LIME feature weights:")
    for feat, w in lime_list[:6]:
        print(f"  {feat:40s} {w:+.4f}")

    # Bar chart of LIME weights
    feats_l = [x[0] for x in lime_list]
    weights  = [x[1] for x in lime_list]
    colors_l = ['#55A868' if w > 0 else '#C44E52' for w in weights]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(feats_l[::-1], weights[::-1], color=colors_l[::-1],
            edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', lw=1.2)
    ax.set_xlabel('LIME weight')
    ax.set_title(f'LIME Local Explanation : Instance {instance_idx}\n'
                 f'True: {cancer.target_names[true_label]}, '
                 f'Predicted: {cancer.target_names[pred_label]}',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("LIME not available : skipping.")

In [ ]:
unittests.exercise_5(compute_lime_explanation)

<a name='7'></a>
## 7. From-Scratch Implementation

Understanding the algorithm behind permutation importance deepens your intuition and allows you to adapt it to non-standard scenarios (custom metrics, grouped permutations, etc.).

<a name='ex-6'></a>
### Exercise 6: Permutation Importance from Scratch

**Your Task:**

Implement `permutation_importance_scratch` that:
1. Computes the baseline score `score(model, X_test, y_test)` using the provided `scoring_fn`.
2. For each feature column (by index), shuffles **only that column** in a copy of `X_test` (leave all others unchanged).
3. Computes the score on the shuffled data.
4. Records `importance = baseline_score - shuffled_score`.
5. Returns a `pd.DataFrame` with columns `feature`, `importance`, sorted descending.

**Arguments:**
- `model` : fitted estimator.
- `X_test` : `pd.DataFrame` test features.
- `y_test` : true labels.
- `scoring_fn` : callable `(model, X, y) → float`, e.g. `accuracy_score`.
- `random_state` : int, for reproducibility.

**Returns:** `pd.DataFrame` with columns `['feature', 'importance']`

In [ ]:
# GRADED FUNCTION: permutation_importance_scratch
def permutation_importance_scratch(model, X_test, y_test, scoring_fn, random_state=42):
    """
    Compute permutation importance from scratch.

    Parameters
    ----------
    model       : fitted estimator with .predict()
    X_test      : pd.DataFrame
    y_test      : array-like
    scoring_fn  : callable (model, X, y) -> float
    random_state: int

    Returns
    -------
    pd.DataFrame  columns=['feature', 'importance'], sorted descending
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###
    return result

In [ ]:
# Test permutation_importance_scratch
# Use the Random Forest trained in Exercise 3
rf_ex6 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_ex6.fit(X_clf_train, y_clf_train)

def acc_fn(mdl, X, y):
    return accuracy_score(y, mdl.predict(X))

scratch_df = permutation_importance_scratch(
    rf_ex6, X_clf_test, y_clf_test, scoring_fn=acc_fn, random_state=42
)

# Compare scratch vs sklearn
sklearn_result = permutation_importance(
    rf_ex6, X_clf_test, y_clf_test, n_repeats=1, random_state=42
)
sklearn_df = pd.DataFrame({
    'feature':    list(X_clf_test.columns),
    'sk_importance': sklearn_result.importances_mean
}).sort_values('sk_importance', ascending=False).reset_index(drop=True)

# Rank correlation between methods
from scipy.stats import spearmanr
merged = scratch_df.merge(sklearn_df, on='feature')
corr, pval = spearmanr(merged['importance'], merged['sk_importance'])
print(f"Spearman rank correlation (scratch vs sklearn): {corr:.4f}  (p={pval:.4f})")

print("\nTop 10 by scratch:")
print(scratch_df.head(10).to_string(index=False))

# Side-by-side bars
top10_s = scratch_df.head(10)
top10_k = sklearn_df.head(10)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df_, title, col in [
        (axes[0], top10_s, 'Permutation Importance (Scratch)', '#4C72B0'),
        (axes[1], top10_k, 'Permutation Importance (sklearn)', '#55A868')]:
    ax.barh(df_['feature'].iloc[::-1],
            df_.iloc[::-1, 1],
            color=col, alpha=0.85, edgecolor='white')
    ax.set_xlabel('Importance (accuracy drop)')
    ax.set_title(title, fontweight='bold')
plt.suptitle('Scratch vs sklearn Permutation Importance Comparison',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_6(permutation_importance_scratch)

<a name='8'></a>
## 8. Conclusion

**Congratulations!** You have completed the Model Interpretation & Explainability assignment.

### Key Takeaways

| Exercise | Technique | What you learned |
|---|---|---|
| 1 | Ridge coefficients | Standardised coefficients directly show feature impact direction and magnitude |
| 2 | Decision tree rules | Every prediction can be traced as a readable if/else path |
| 3 | Permutation importance | Model-agnostic, test-set importance that avoids MDI bias |
| 4 | SHAP TreeExplainer | Exact, additive Shapley values for global + local explanation |
| 5 | LIME | Fast local linear approximation for any black-box model |
| 6 | Permutation (scratch) | Core algorithm: shuffle one feature, measure score drop |

### When to Use What

```
Need to explain the model globally to the team?
  → SHAP bar plot (mean |SHAP value|) or permutation importance

Need to explain a single prediction (e.g. loan rejection)?
  → SHAP force plot or LIME explanation

Working with a linear model?
  → Standardised coefficients are exact and sufficient

Need fast, model-agnostic importances?
  → Permutation importance (1 repeat) : seconds, not minutes
```

### Installation Note

If SHAP or LIME were unavailable, install them:
```bash
pip install shap lime
```